# Predicting Adverse Interaction Signals in Mental-Health Polypharmacy Using Machine Learning

**Course:** HADM 283 - AI in Healthcare, Hofstra University  
**Authors:** Desray K. Desir & TaraJee Clarke  

### Project Overview
This notebook predicts PRR-based adverse drug-drug interaction (DDI) signals among mental-health drug pairs using the **TwoSIDES** dataset (derived from FDA FAERS). The pipeline is split into two phases:

1. **Data Engineering** — chunked loading, Polars filtering, label creation, subset export
2. **Modeling** — EDA, logistic regression (baseline + balanced), random forest, evaluation

### Dataset
Download TwoSIDES from: https://tatonettilab.org/resources/nsides/

## Part 1 — Data Engineering & Scaling

### 1.1 Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, roc_curve
)
import pickle, warnings
warnings.filterwarnings('ignore')
print('Libraries loaded.')

### 1.2 Mental-Health Drug Keyword List

In [ ]:
MENTAL_HEALTH_KEYWORDS = [
    # SSRIs / SNRIs
    'fluoxetine','sertraline','escitalopram','citalopram','paroxetine',
    'fluvoxamine','venlafaxine','duloxetine','desvenlafaxine','levomilnacipran',
    # Antidepressants (other)
    'bupropion','mirtazapine','trazodone','nefazodone','amitriptyline',
    'nortriptyline','imipramine','clomipramine',
    # Antipsychotics
    'quetiapine','olanzapine','risperidone','aripiprazole','ziprasidone',
    'haloperidol','clozapine','lurasidone','paliperidone','asenapine',
    # Mood stabilizers
    'lithium','valproate','valproic acid','lamotrigine','carbamazepine','oxcarbazepine',
    # Benzodiazepines
    'alprazolam','diazepam','clonazepam','lorazepam','temazepam','oxazepam','triazolam','midazolam',
    # Stimulants
    'methylphenidate','amphetamine','lisdexamfetamine','dextroamphetamine','atomoxetine','modafinil',
    # Sleep / other
    'zolpidem','eszopiclone','buspirone','hydroxyzine',
]

def matches_mental_health(drug_name):
    if pd.isna(drug_name): return False
    drug_lower = str(drug_name).lower()
    return any(kw in drug_lower for kw in MENTAL_HEALTH_KEYWORDS)

print(f'Keyword list: {len(MENTAL_HEALTH_KEYWORDS)} drugs')

### 1.3 Load 100K-Row Chunk (pandas → intermediate CSV)

In [ ]:
# Update this path to wherever you saved the TwoSIDES file
TWOSIDES_PATH = 'twosides.csv'

df_chunk = pd.read_csv(TWOSIDES_PATH, nrows=100_000, low_memory=False)
print(f'Chunk shape: {df_chunk.shape}')
df_chunk.head()

In [ ]:
# Save intermediate CSV (avoids encoding errors when loading into Polars)
df_chunk.to_csv('twosides_chunk_100k.csv', index=False)
print('Intermediate file saved: twosides_chunk_100k.csv')

### 1.4 Load into Polars (relaxed parsing)

In [ ]:
df_polars = pl.read_csv(
    'twosides_chunk_100k.csv',
    encoding='utf8-lossy',
    ignore_errors=True,
    truncate_ragged_lines=True,
    infer_schema_length=10_000,
)
print(f'Polars shape: {df_polars.shape}')
df_polars.head()

### 1.5 Filter to Mental-Health Drug Pairs

In [ ]:
df_pd = df_polars.to_pandas()

mask = (
    df_pd['drug_1_concept_name'].apply(matches_mental_health) |
    df_pd['drug_2_concept_name'].apply(matches_mental_health)
)

df_mental = df_pd[mask].copy()
print(f'Mental-health subset: {len(df_mental):,} rows')
print(f'Unique drug_1: {df_mental["drug_1_concept_name"].nunique()}')
print(f'Unique drug_2: {df_mental["drug_2_concept_name"].nunique()}')

### 1.6 PRR Conversion + Binary Label Creation

In [ ]:
df_mental['PRR_float'] = pd.to_numeric(df_mental['PRR'], errors='coerce')
df_mental.dropna(subset=['PRR_float'], inplace=True)

# PRR > 1 → elevated signal (label 1); PRR ≤ 1 → no signal (label 0)
df_mental['label'] = (df_mental['PRR_float'] > 1).astype(int)

print('Label distribution:')
print(df_mental['label'].value_counts())

In [ ]:
df_mental.to_csv('twosides_mental_health_subset.csv', index=False)
print('Subset saved: twosides_mental_health_subset.csv')

---
## Part 2 — EDA, Modeling & Evaluation

### 2.1 Load Prepared Subset

In [ ]:
df = pd.read_csv('twosides_mental_health_subset.csv')
print(f'Loaded: {df.shape}')
df.head()

### 2.2 Exploratory Data Analysis

In [ ]:
print(df['PRR_float'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('EDA — Mental-Health DDI Subset', fontsize=14, fontweight='bold')

# Class distribution
label_counts = df['label'].value_counts().sort_index()
axes[0].bar(['No Signal (0)', 'Signal (1)'], label_counts.values,
            color=['#4c72b0', '#dd8452'], edgecolor='white')
axes[0].set_title('Label Distribution (Class Imbalance)')
axes[0].set_ylabel('Count')

# PRR distribution
axes[1].hist(df['PRR_float'].clip(upper=20), bins=50, color='#55a868', edgecolor='white', alpha=0.85)
axes[1].axvline(x=1.0, color='red', linestyle='--', linewidth=1.5, label='PRR = 1 threshold')
axes[1].set_title('PRR Distribution (clipped at 20)')
axes[1].set_xlabel('PRR Value')
axes[1].legend()

plt.tight_layout()
plt.show()

### 2.3 Feature Preparation & Train/Test Split

In [ ]:
features = ['drug_1_concept_name', 'drug_2_concept_name']
df_model = df[features + ['label']].dropna()

X = df_model[features]
y = df_model['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'Train label distribution:\n{y_train.value_counts()}')

### 2.4 Train Models

In [ ]:
def build_pipeline(classifier):
    return Pipeline([
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ('model', classifier)
    ])

models = {
    'LR Baseline':            build_pipeline(LogisticRegression(max_iter=500, random_state=42)),
    'LR Balanced':            build_pipeline(LogisticRegression(max_iter=500, class_weight='balanced', random_state=42)),
    'Random Forest Balanced': build_pipeline(RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)),
}

for name, pipeline in models.items():
    print(f'Training: {name}...')
    pipeline.fit(X_train, y_train)

print('\nAll models trained.')

### 2.5 Evaluate Models — Classification Reports & Confusion Matrices

In [ ]:
results = []

for name, pipeline in models.items():
    y_pred  = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    print(f'\n── {name} ──')
    print(f'  Accuracy: {acc:.4f} | ROC AUC: {auc:.4f}')
    print(classification_report(y_test, y_pred, target_names=['No Signal', 'Signal']))

    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm, display_labels=['No Signal', 'Signal']).plot(cmap='Blues', ax=ax)
    ax.set_title(f'Confusion Matrix — {name}')
    plt.tight_layout()
    plt.show()

    results.append({'Model': name, 'Accuracy': round(acc, 4), 'ROC AUC': round(auc, 4)})

### 2.6 ROC Curve Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, pipeline in models.items():
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
ax.set_title('ROC Curves — All Models')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 2.7 Model Comparison Summary

In [ ]:
summary = pd.DataFrame(results)
print(summary.to_string(index=False))

### 2.8 Error Analysis

In [ ]:
best = models['LR Balanced']
y_pred_best = best.predict(X_test)

error_df = X_test.copy()
error_df['true_label']      = y_test.values
error_df['predicted_label'] = y_pred_best

fp = error_df[(error_df['true_label'] == 0) & (error_df['predicted_label'] == 1)]
fn = error_df[(error_df['true_label'] == 1) & (error_df['predicted_label'] == 0)]

print(f'False Positives: {len(fp)}')
print(fp.head(10).to_string(index=False))

print(f'\nFalse Negatives: {len(fn)}')
print(fn.head(10).to_string(index=False))

### 2.9 Save Best Model

In [ ]:
with open('lr_balanced_pipeline.pkl', 'wb') as f:
    pickle.dump(models['LR Balanced'], f)
print('Model saved: lr_balanced_pipeline.pkl')

---
## Results Summary

| Model | Accuracy | ROC AUC |
|-------|----------|---------|
| LR Baseline | 0.934 | 0.690 |
| LR Balanced | 0.612 | 0.691 |
| Random Forest Balanced | 0.612 | 0.691 |

**Key findings:**
- Baseline LR achieves high accuracy by predicting the majority class (label 1) almost exclusively — a misleading metric under heavy class imbalance.
- Balanced models trade overall accuracy for improved minority class recall, which is clinically preferable.
- Random Forest shows no advantage over LR here because one-hot encoded drug names create a sparse, identity-based feature space — no nonlinear structure for the forest to exploit.
- **Future work:** drug class/mechanism features, SHAP interpretability, XGBoost, DrugCentral integration.